# Rural Road Extraction - Kaggle Pipeline Runner

This notebook initializes the DeepGlobe dataset with your generated weak labels, and runs the `MobileViT v2` training pipeline using AMP and dynamic topology-aware clDice loss.

In [ ]:
import os
import sys

# Add the project root to Python path so we can import 'src'
# Assuming this notebook is run from the project root or backend directory
if os.path.exists('backend'):
    sys.path.append('backend')
elif os.path.exists('src'):
    sys.path.append('.')

# Kaggle Paths Setup
# ---------------------------------------------------------
TRAIN_IMG_DIR = '/kaggle/input/deepglobe-road-extraction-dataset/train'
TRAIN_MASK_DIR = '/kaggle/working/osm_weak_labels'

# Validation split (Assuming validation data is in the same deepglobe dataset under 'valid')
VAL_IMG_DIR = '/kaggle/input/deepglobe-road-extraction-dataset/valid'
VAL_MASK_DIR = '/kaggle/input/deepglobe-road-extraction-dataset/valid' # Update if val masks are elsewhere

OUTPUT_DIR = '/kaggle/working/models'

# Path validation
print("--- Validating Kaggle Paths ---")
for name, path in [('Train Images', TRAIN_IMG_DIR), ('Train Masks', TRAIN_MASK_DIR)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"\u274c Path not found for {name}: {path}\n" 
                                f"Please check your Kaggle inputs and working directory.")
    else:
        print(f"\u2705 {name} path exists: {path}")
print("Paths verified successfully!\n")

## Execute Training Loop

We use the updated `train.py` script which supports independent directories for images and masks.

In [ ]:
# Set W&B API Key (Uncomment and replace if logging to Weights & Biases)
# os.environ["WANDB_API_KEY"] = "your_wandb_api_key_here"

# Configuration
EPOCHS = 50
BATCH_SIZE = 16
LR = 1e-3
NUM_WORKERS = 4
RUN_NAME = "kaggle-mobilevit-v2-run1"

# Run the MLOps pipeline script
!python backend/scripts/train.py \
    --train_image_dir {TRAIN_IMG_DIR} \
    --train_mask_dir {TRAIN_MASK_DIR} \
    --val_image_dir {VAL_IMG_DIR} \
    --val_mask_dir {VAL_MASK_DIR} \
    --output_dir {OUTPUT_DIR} \
    --run_name {RUN_NAME} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --num_workers {NUM_WORKERS}

print("\n\u2728 Training completed! Best model saved to:", os.path.join(OUTPUT_DIR, "best_model.pth"))